# Derm ReasonEdit — Edit & Eval (given user, given VLM)

Runs the **top-1 reliability** experiment end-to-end for one `(user, model)` pair:

1. **Build edit set** = the model's top-1 *errors* (so baseline top-1 acc should be ~0).
2. **Baseline** — ask the same **top-3 differential** question as the annotation interface; extract top-1 for scoring.
3. **Edit** — codebook value is `New Fact: The most likely differential is {human top-1}.` (`dual_sbert`, key = `<image, question>`).
4. **Apply retrieval + re-ask** the same top-3 question; extract post-edit top-1 (should flip toward the human answer).
5. **Report** reliability before vs after + qualitative cards, and save a results JSON.

This is the notebook twin of `run_efficacy.py` — it imports the same helpers so results match the CLI/sbatch path.

> **Env:** `dermedit` (Python 3.10), created from the repo `readme.md`. This run stays on `dual_sbert` with `lang_scaler_sbert: 1`; it does not use the tuned scaler.
> Both models are already cached on scratch (`XDG_CACHE_HOME`), so no download. Set `HF_TOKEN` for gated repos.

## 0. Switches — pick the user and the VLM

In [ ]:
import sys
from pathlib import Path

# Locate the reliability package regardless of the kernel's cwd (JupyterLab often
# starts the kernel at the server root, not the notebook folder).
def _find_pkg():
    hard = Path("/scratch/jq2uw/MME/derm_vlm_edit/reliability")
    for p in [Path.cwd(), Path.cwd().parent, hard, *Path.cwd().parents]:
        if (p / "config.py").exists() and p.name == "reliability":
            return p
        cand = p / "derm_vlm_edit" / "reliability"
        if (cand / "config.py").exists():
            return cand
    return hard

PKG = _find_pkg()
assert (PKG / "config.py").exists(), f"reliability/config.py not found (looked near {PKG})"
sys.path.insert(0, str(PKG))
print("package root:", PKG)

import config as C
sys.path.insert(0, str(C.DERM_EDIT_ROOT))

# ---- experiment switches -------------------------------------------------
USER           = "jesse.y.liu.md@gmail.com"   # "jesse.y.liu.md@gmail.com" | "achiou@stanford.edu" | "lhunterellul"
MODEL_SHORT    = "medgemma "          # "medgemma" | "dermatollama"
RATIONALE_MODE = C.RATIONALE_MODE    # "edited_only" (human-authored) | "final"
NORM_ANSWER    = "y16"       # "y16" (class flip) | "human" (exact canonical text)
SUBSAMPLE      = 0                     # 0 = use all edits; else cap for a quick smoke run
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS      # 512 — top-3 + reasoning, same as collection

ANNOT_MODEL = C.MODEL_SHORT_TO_ANNOT[MODEL_SHORT]
print(f"user   = {USER}")
print(f"model  = {MODEL_SHORT}  (annotation label: {ANNOT_MODEL})")
print(f"rationale_mode = {RATIONALE_MODE} | norm_answer = {NORM_ANSWER} | subsample = {SUBSAMPLE} | max_new_tokens = {MAX_NEW_TOKENS}")
print(f"annotations exists: {C.ANNOTATIONS_CSV.exists()} | images exists: {C.IMAGE_ROOT.exists()}")

package root: /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability
user   = jesse.y.liu.md@gmail.com
model  = medgemma  (annotation label: MedGemma)
rationale_mode = edited_only | norm_answer = y16 | subsample = 0 | max_new_tokens = 512
annotations exists: True | images exists: True


## 1. Build the edit set (model's top-1 errors for this user)

In [2]:
import json
from data_engineering.build_edit_set import build as build_edit_set

edits, summary = build_edit_set(USER, ANNOT_MODEL, RATIONALE_MODE, write=True, norm_answer=NORM_ANSWER)
if SUBSAMPLE and len(edits) > SUBSAMPLE:
    edits = edits[:SUBSAMPLE]

assert edits, f"No usable edits for user={USER} model={ANNOT_MODEL}."
print(json.dumps({k: v for k, v in summary.items() if not k.endswith("_counts")}, indent=2))
print(f"\n{len(edits)} edits -> {C.edit_set_path(USER, ANNOT_MODEL)}")

{
  "user": "jesse.y.liu.md@gmail.com",
  "model": "MedGemma",
  "rationale_mode": "edited_only",
  "norm_answer": "y16",
  "question": "You are an expert dermatologist. Please give the top 3 diagnoses in your differential, and provide reasoning for each diagnosis, in format:\n1. [Diagnosis]: [Reasoning]\n2. [Diagnosis]: [Reasoning]\n3. [Diagnosis]: [Reasoning]",
  "fact_template": "New Fact: The top 1 most likely differential is {answer}.",
  "n_marked_incorrect": 25,
  "n_edits": 24,
  "n_dropped_same_grain": 1,
  "n_images_missing": 0,
  "n_multi_dx_top1": 0,
  "n_y16_unmapped": 0,
  "n_empty_rationale": 0
}

24 edits -> /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability/results/edit_set__jesse_y_liu_md_at_gmail_com__MedGemma.jsonl


## 2. Load the VLM + resolve ReasonEdit layer names
First model load downloads weights (can be slow / needs `HF_TOKEN` for gated repos).

In [3]:
import torch

sys.path.insert(0, str(C.DERM_EDIT_ROOT))
from reasonedit import ReasonEdit, VQAModel, load_config

# reuse the exact helpers from the CLI runner so notebook == run_efficacy.py
from run_efficacy import is_correct, resolve_layers
from derm_dataset import DermEditDataset

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# Editor defaults (in ReasonEdit.__init__) already match rankedit answer-only IKE:
# key_type=answer, merge_keys=False, hubness_keys=False, cap_keys=1, n_radius_samples=0.
config = load_config(
    MODEL_SHORT, device=device,
    batch_size=1, max_new_tokens=MAX_NEW_TOKENS, seed=42,
)
config.editor.fact_template = C.FACT_TEMPLATE

model = VQAModel(config)
resolve_layers(model, config)   # corrects best-guess inner_params_* against the loaded tree
print("device:", device)
print("fact_template:", C.FACT_TEMPLATE)
print("editor defaults: key_type/merge/hubness/cap_keys live in ReasonEdit.__init__")

/scratch/jq2uw/conda_ex/conda_venv/dermedit/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.41it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[layers] inner_params_vision: ['model.vision_tower.vision_model.encoder.layers.24.mlp.fc1.weight']
[layers] inner_params_lang: ['model.language_model.layers.20.mlp.gate_proj.weight']
[layers] inner_params: ['model.language_model.layers.30.mlp.gate_proj.weight']
device: cuda:0
fact_template: New Fact: The top 1 most likely differential is {answer}.
editor defaults: key_type/merge/hubness/cap_keys live in ReasonEdit.__init__


### 2.5 Sanity — confirm the ReasonEdit vision hook layer resolves (GPU)
`dual_sbert` only hooks `inner_params_vision[0]`. `resolve_layers` (cell above) already mapped the
prefix-free yaml name to the exact live param (Mllama is a PeftModel, so names carry a
`base_model.model.` prefix). This cell verifies that resolution against the loaded model.

In [4]:
import re
from reasonedit.utils import validate_and_correct_param_name

params = dict(model.model.named_parameters())
resolved = config.model.inner_params_vision[0]   # already corrected by resolve_layers
print("inner_params_vision ->", resolved)
print("exists in model     :", resolved in params,
      "| shape:", tuple(params[resolved].shape) if resolved in params else None)

# show the tail of the vision-MLP weight list so you can pick a different late block if desired
vis = sorted(n for n in params if re.search(r"vision.*mlp\.(fc1|fc2|up_proj|down_proj)\.weight$", n))
print(f"\n{len(vis)} vision-mlp weight params; last 5:")
for n in vis[-5:]:
    print("  ", n)

inner_params_vision -> model.vision_tower.vision_model.encoder.layers.24.mlp.fc1.weight
exists in model     : True | shape: (4304, 1152)

54 vision-mlp weight params; last 5:
   model.vision_tower.vision_model.encoder.layers.7.mlp.fc2.weight
   model.vision_tower.vision_model.encoder.layers.8.mlp.fc1.weight
   model.vision_tower.vision_model.encoder.layers.8.mlp.fc2.weight
   model.vision_tower.vision_model.encoder.layers.9.mlp.fc1.weight
   model.vision_tower.vision_model.encoder.layers.9.mlp.fc2.weight


## 3. Baseline top-3 generation (pre-edit; eval extracts top-1)

In [ ]:
ds = DermEditDataset(config, edits)
ds.set_dataloader()

print("generating pre-edit top-3 (eval extracts top-1) ...")
ds.task_generate(model, use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
pre = [ex.get("pred", {}).get("answer", "") for ex in ds.data]
pre_raw = [ex.get("pred", {}).get("raw", "") for ex in ds.data]
for ex, pa in list(zip(ds.data, pre))[:5]:
    print(f"  {ex['case_id']:>16}  pred={pa!r:40}  human={ex['answer']!r}")

generating pre-edit top-3 (eval extracts top-1) ...


Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


In [ ]:
import torch
from PIL import Image

ex = ds.data[0]
img = ex["image"]
img = Image.open(img).convert("RGB") if isinstance(img, str) else img

inputs = model.preprocess([img], [ex["prompt"]], model.processor)   # CPU tensors
ids = inputs["input_ids"]
emb = model.model.get_input_embeddings()
cfg = model.model.config
img_tok = getattr(cfg, "image_token_index", None) or getattr(cfg, "image_token_id", None)

print("keys        :", list(inputs.keys()))
print("input_ids   :", tuple(ids.shape), "min", int(ids.min()), "max", int(ids.max()))
print("num_embeds  :", emb.num_embeddings)
print("TOKEN OOB?  :", int(ids.max()) >= emb.num_embeddings)      # <-- the smoking gun
if "pixel_values" in inputs:
    print("pixel_values:", tuple(inputs["pixel_values"].shape))
if img_tok is not None:
    print("image_tok id:", img_tok, "count in ids:", int((ids == img_tok).sum()))

keys        : ['input_ids', 'attention_mask', 'pixel_values', 'aspect_ratio_ids', 'aspect_ratio_mask', 'cross_attention_mask']
input_ids   : (1, 69) min 11 max 128256
num_embeds  : 128264
TOKEN OOB?  : False
pixel_values: (1, 1, 4, 3, 560, 560)
image_tok id: 128256 count in ids: 1


## 4. Edit (build codebook) → apply retrieval → regenerate

In [ ]:
print("building ReasonEdit codebook ...")
editor = ReasonEdit(config, model)
editor.edit(config, edit_ds=ds)
editor.apply_to_dataset(ds)

print("generating post-edit top-3 (eval extracts top-1) ...")
ds.task_generate(model, use_cache=True, max_new_tokens=MAX_NEW_TOKENS)
post = [ex.get("pred", {}).get("answer", "") for ex in ds.data]
post_raw = [ex.get("pred", {}).get("raw", "") for ex in ds.data]

retr_log = {r["uid"]: r.get("facts", []) for r in (editor.last_retrieval_log or [])}
for ex, qa in list(zip(ds.data, post))[:5]:
    print(f"  {ex['case_id']:>16}  post={qa!r:40}  human={ex['answer']!r}")

building ReasonEdit codebook ...
[ReasonEdit] mode=dual_sbert, key_type=answer, pair_rationale_w=orig, merge=False, hubness=False, cap_keys=1
[ReasonEdit] edit 1/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 2/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 3/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 4/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 5/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 6/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 7/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 8/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 9/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 10/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 11/14...[Keys] +1 added, 0 merged (from 1 candidates)
[ReasonEdit] edit 12/14...[Keys] +1 added, 0 merged (from 1 candidates)
[Re

## 5. Reliability before vs after (the 0 → 1 flip)

In [ ]:
rows = []
n_pre = n_post = 0
for ex, pa, qa, pa_raw, qa_raw in zip(ds.data, pre, post, pre_raw, post_raw):
    hn, hy = ex["answer"], ex.get("human_top1_y16")
    pc, qc = is_correct(pa, hn, hy, NORM_ANSWER), is_correct(qa, hn, hy, NORM_ANSWER)
    n_pre += int(pc); n_post += int(qc)
    rows.append({
        "uid": ex["uid"], "case_id": ex.get("case_id"),
        "model_top1": ex.get("model_top1"),
        "human_top1": ex.get("human_top1_raw"), "human_norm": hn, "human_y16": hy,
        "pred_pre": pa, "pred_post": qa,
        "pred_pre_raw": pa_raw, "pred_post_raw": qa_raw,
        "correct_pre": pc, "correct_post": qc,
        "retrieved_facts": retr_log.get(ex["uid"], []),
    })

n = len(rows)
n_pre = C.cap_before(n_pre)
result = {
    "user": USER, "model": ANNOT_MODEL, "short_model": MODEL_SHORT,
    "rationale_mode": RATIONALE_MODE, "norm_answer": NORM_ANSWER, "question": C.QUESTION,
    "fact_template": C.FACT_TEMPLATE,
    "n_edits": n,
    "reliability_pre": n_pre / n if n else 0.0,
    "reliability_post": n_post / n if n else 0.0,
    "n_correct_pre": n_pre, "n_correct_post": n_post,
    "inner_params_vision": getattr(config.model, "inner_params_vision", None),
    "cases": rows,
}
print(f"reliability_pre  = {result['reliability_pre']:.3f} ({n_pre}/{n})")
print(f"reliability_post = {result['reliability_post']:.3f} ({n_post}/{n})")
print(f"flipped wrong->right: {sum(r['correct_post'] and not r['correct_pre'] for r in rows)}")

reliability_pre  = 0.000 (0/14)
reliability_post = 1.000 (14/14)
flipped wrong->right: 14


## 6. Per-case table + before/after cards

In [ ]:
for r in rows:
    flag = "OK " if r["correct_post"] else "-- "
    chg  = "  (flip)" if r["correct_post"] and not r["correct_pre"] else ""
    print(f"{flag}{r['case_id']:>16} | model:{str(r['model_top1'])[:22]:22} | "
          f"pre:{str(r['pred_pre'])[:22]:22} -> post:{str(r['pred_post'])[:22]:22} | "
          f"human:{r['human_top1']}{chg}")

OK    1025_combined | model:Basal Cell Carcinoma   | pre:Basal Cell Carcinoma   -> post:Lichenoid Keratosis    | human:Lichenoid keratosis  (flip)
OK    1041_combined | model:Squamous Cell Carcinom | pre:Squamous Cell Carcinom -> post:Lichenoid Keratosis    | human:Lichenoid keratosis  (flip)
OK     105_combined | model:Nevus                  | pre:Nevus                  -> post:Melanoma in situ       | human:Melanoma in situ  (flip)
OK    1100_combined | model:Nevus                  | pre:Nevus                  -> post:Epidermoid cyst        | human:Epidermoid cyst  (flip)
OK    1160_combined | model:Dermatofibroma         | pre:Nevus                  -> post:Lichenoid Keratosis    | human:Lichenoid keratosis  (flip)
OK     134_combined | model:Squamous Cell Carcinom | pre:Basal Cell Carcinoma   -> post:Eczematous Dermatitis  | human:Eczematous dermatitis  (flip)
OK     151_combined | model:Basal Cell Carcinoma   | pre:Basal Cell Carcinoma   -> post:Lichenoid Keratosis    | human:Lich

In [ ]:
# import textwrap
# import matplotlib.pyplot as plt
# from PIL import Image

# def show_cards(edits, rows, n=6):
#     n = min(n, len(edits))
#     fig, axes = plt.subplots(n, 2, figsize=(14, 3.3 * n), gridspec_kw={"width_ratios": [1, 2.6]})
#     if n == 1:
#         axes = axes.reshape(1, 2)
#     for i, (e, r) in enumerate(zip(edits[:n], rows[:n])):
#         try:
#             axes[i, 0].imshow(Image.open(e["image"]).convert("RGB"))
#         except Exception as exc:
#             axes[i, 0].text(0.5, 0.5, f"no image\n{exc}", ha="center")
#         axes[i, 0].axis("off")
#         axes[i, 0].set_title(("OK " if r["correct_post"] else "-- ") + str(e["case_id"]), fontsize=9)
#         facts = "\n".join(f"  - {f}" for f in r["retrieved_facts"]) or "  (none)"
#         txt = (
#             f"MODEL top-1 (wrong): {r['model_top1']}\n"
#             f"PRE  pred          : {r['pred_pre']}\n"
#             f"POST pred          : {r['pred_post']}\n"
#             f"HUMAN top-1        : {r['human_top1']}  (y16: {r['human_y16']})\n\n"
#             f"retrieved facts:\n{facts}\n\n"
#             f"human reasoning:\n{textwrap.fill(e.get('rationale', ''), 92)}"
#         )
#         axes[i, 1].axis("off")
#         axes[i, 1].text(0.0, 1.0, txt, va="top", ha="left", fontsize=8.5, family="monospace")
#     plt.tight_layout(); plt.show()

# show_cards(edits, rows, n=6)

## 7. Save results JSON (same schema as `run_efficacy.py`)

In [ ]:
import os

os.makedirs(C.RESULTS_DIR, exist_ok=True)
safe_user = USER.replace("@", "_at_").replace(".", "_")
out = C.RESULTS_DIR / f"reliability__{safe_user}__{MODEL_SHORT}.json"
with open(out, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)
print("saved ->", out)

saved -> /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability/results/reliability__jesse_y_liu_md_at_gmail_com__dermatollama.json


## 8. Collect ALL runs → PI-ready before/after table
Run this **after** you've executed the notebook for each `(user, vlm)` combo (each writes its own
`results/reliability__*.json`). This cell scans them all and builds the 3×2 table
(Markdown + LaTeX + CSV in `results/`). It does not need the model loaded.

In [ ]:
import importlib
import make_report as MR
importlib.reload(MR)
from IPython.display import Markdown, display

results = MR.load_results(C.RESULTS_DIR)
found = sorted(f"{u}/{sm}" for (u, sm) in results)
print(f"found {len(results)} runs:", found)

users = [u for u in MR.REPORT_USERS if any((u, sm) in results for sm, _ in MR.VLMS)] \
        or sorted({u for (u, _) in results})
rows, pooled = MR.build_rows(results, users, MR.VLMS)

md = MR.render_markdown(rows, pooled, MR.VLMS)
out = C.RESULTS_DIR / "report.csv"
MR.render_csv(rows, pooled, MR.VLMS, out)
print("wrote", out, "\n")

display(Markdown(md))

found 6 runs: ['achiou@stanford.edu/dermatollama', 'achiou@stanford.edu/medgemma', 'jesse.y.liu.md@gmail.com/dermatollama', 'jesse.y.liu.md@gmail.com/medgemma', 'lhunterellul/dermatollama', 'lhunterellul/medgemma']
wrote /sfs/weka/scratch/jq2uw/MME/derm_vlm_edit/reliability/results/report.csv 



| User | MedGemma Before | MedGemma After | MedGemma Δ | DermatoLlama Before | DermatoLlama After | DermatoLlama Δ |
|---|---|---|---|---|---|---|
| jesse.y.liu.md | 0% (0/24) | 100% (24/24) | +100 | 0% (0/14) | 100% (14/14) | +100 |
| achiou | 0% (0/20) | 95% (19/20) | +95 | 0% (0/13) | 100% (13/13) | +100 |
| lhunterellul | 0% (0/8) | 100% (8/8) | +100 | 33% (2/6) | 100% (6/6) | +67 |
| **Pooled** | 0% (0/52) | 98% (51/52) | +98 | 6% (2/33) | 100% (33/33) | +94 |
